# 🎯 Churn Prediction Modeling Engine

## Model Suite:
- **Logistic Regression** (Interpretable Linear Baseline with Balanced Weights)
- **XGBoost** (Gradient Boosted Trees with Scale-Pos-Weight)
- **LightGBM** (Histogram-based Boosting with Optuna Tuning)
- **CatBoost** (Ordered Boosting with Auto-Balanced Class Weights)

## Key Techniques:
- Stratified Train / Validation / Test split (70 / 15 / 15).
- Evaluation metrics: ROC-AUC, PR-AUC, Brier Score, Precision, Recall.
- Platt Scaling (Sigmoid Probability Calibration).
- MLflow Experiment Tracking.

In [1]:
import os, sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

from src.models.train_churn import train_and_evaluate_churn_models
from src.models.predict import predict_churn
print('Environment ready.')

In [2]:
# Inspect Benchmark Results
with open('models/churn_metadata.json', 'r') as f:
    meta = json.load(f)

print(f'Selected Best Model: {meta["best_model_name"]}')
benchmarks = meta['all_benchmarks']

summary_rows = []
for model_name, evals in benchmarks.items():
    summary_rows.append({
        'Model': model_name,
        'Val ROC-AUC': evals['validation']['roc_auc'],
        'Val PR-AUC': evals['validation']['pr_auc'],
        'Test ROC-AUC': evals['test']['roc_auc'],
        'Test PR-AUC': evals['test']['pr_auc'],
        'Test Brier Score': evals['test']['brier_score'],
        'Test F1': evals['test']['f1']
    })
bench_df = pd.DataFrame(summary_rows).sort_values('Test PR-AUC', ascending=False)
display(bench_df)

In [3]:
# Test Real-Time Single-Customer Prediction
sample_input = {
    'tenure': 2,
    'monthly_charges': 89.50,
    'total_charges': 179.0,
    'contract_Month-to-month': 1,
    'internet_service_Fiber optic': 1,
    'payment_method_Electronic check': 1
}
res = predict_churn(sample_input)
print('Prediction Output for High-Risk Test Account:')
print(json.dumps(res, indent=2))